# Objective:
# Analyze customer behavior to identify the key factors contributing to customer churn.
# Build predictive machine learning models to classify customers who are likely to churn,
# enabling proactive customer retention strategies.

***Loading of Data***

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv(r"C:\Users\ADMIN\Downloads\ML_Practice_DATA\IBM teleco_churn\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape

***EDA***

# Perform an initial quality assessment by examining:
# • Number of rows and columns
# • Data types
# • Sample records
# • Missing values
# • Duplicate records
#
# This helps identify potential data quality issues before preprocessing.

In [ ]:
df=pd.read_csv(r"C:\Users\ADMIN\Downloads\ML_Practice_DATA\IBM teleco_churn\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape


In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isna().sum()

# Convert TotalCharges from object to numeric.
# Blank values are converted into NaN so they can be handled appropriately.
#
# TotalCharges should be numeric because it represents cumulative billing.

In [ ]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.isna().sum()

# Check missing values after data type conversion.
#
# Missing values in TotalCharges usually belong to customers with very
# short tenure and should be either removed or imputed depending on
# business requirements.

In [ ]:
#we have 11 null value in Total Chage Columns 
#either we have to delete the data or impute the data with null
print(df['TotalCharges'].mean())
#so as it is a small number we will remover these rows from our dataset
df=df.dropna()

In [ ]:
df.head(5)

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns

# Analyze numerical variables such as:
# • Tenure
# • MonthlyCharges
# • TotalCharges
#
# Goal:
# Understand distributions, detect skewness,
# identify outliers, and compare churn vs non-churn customers.

In [ ]:
num_cols = ['tenure','MonthlyCharges','TotalCharges']

for col in num_cols:
    print(df[col].describe())

# Explore the overall distribution of customer attributes
# to understand customer demographics and service usage patterns.

In [ ]:
import seaborn as sns
sns.histplot(df['tenure'])


In [ ]:
sns.boxplot(x=df['tenure'])

***Tenure have no outlier****

In [ ]:
print(sns.histplot(df['MonthlyCharges']))


In [ ]:
print(sns.boxplot(df['MonthlyCharges']))

In [ ]:
print(sns.boxplot(df['TotalCharges']))

In [ ]:
sns.histplot(df['TotalCharges'])

* treatment of  outlier as we will be using Clasisfer model which are not sensitive to outlier so we will ignoure the outlier


# Remove customerID since it is a unique identifier and
# does not contribute to predicting customer churn.

In [ ]:
#customerID is irrelelvent columns as it is not required in model 
df=df.drop(columns=['customerID'])
df.columns
cat_cols = df.select_dtypes(include='object').columns
cat_cols

***Value COunt of all the categorical variable to check hte spred of the data***

In [ ]:
for col in cat_cols:
    print(f"\nValue Counts for {col}")
    print(df[col].value_counts(dropna=False))

***Understand relationship with Churn.***

In [ ]:
sns.boxplot(x='Churn', y='tenure', data=df)

In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)


In [ ]:
sns.boxplot(x='Churn', y='TotalCharges', data=df)

# Analyze categorical variables to identify
# which customer groups experience higher churn.
#
# Examples:
# Contract Type
# Internet Service
# Payment Method
# Gender
# Senior Citizen

In [ ]:

print(pd.crosstab(df['Contract'],df['Churn']))
print(pd.crosstab(df['gender'],df['Churn']))
print(pd.crosstab(df['Partner'],df['Churn']))
print(pd.crosstab(df['Dependents'],df['Churn']))
print(pd.crosstab(df['InternetService'],df['Churn']))
print(pd.crosstab(df['OnlineSecurity'],df['Churn']))
print(pd.crosstab(df['TechSupport'],df['Churn']))
print(pd.crosstab(df['PaymentMethod'],df['Churn']))

# Encode categorical variables into numerical format
# so they can be used by machine learning algorithms.
# Different encoding techniques may be suitable
# depending on the model being used.

In [ ]:
df['Churn']=df['Churn'].map({'Yes':1,'No':0})
df['Churn'].value_counts()

* Convert Categorycal value to integer using Label-encoding 

***Label ENCODING***

In [ ]:
from sklearn.preprocessing import LabelEncoder


In [ ]:
encoder=LabelEncoder()
for col in df.columns:
    if df[col].dtype=='object':
        df[col]=encoder.fit_transform(df[col])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.show()

***#as tenure and monthly charge are highly cooretaled so i am dropping tenure***


In [ ]:
df=df.drop(columns=['tenure'])
df.columns

***Standared Scalling for normalising the data***

* *Build Model*
* logistic Regression
* random forest
* gradient boosting
* Xgboosting

***Model Building***

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.linear_model import LogisticRegression


In [ ]:
x=df.drop(columns=['Churn'])
y=df[['Churn']]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=1)

In [ ]:
num_cols=['MonthlyCharges','TotalCharges']
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])

In [ ]:
model=LogisticRegression(class_weight='balanced')
model.fit(x_train,y_train)
pred=model.predict(x_test)
acc=accuracy_score(y_test,pred)
print(acc)
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

In [ ]:
#DecisionTree
from sklearn.tree import DecisionTreeClassifier
model1=DecisionTreeClassifier(class_weight='balanced')
model1.fit(x_train,y_train)
pred=model1.predict(x_test)
acc=accuracy_score(y_test,pred)
print(acc)
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))


In [ ]:
#RandomForest
from sklearn.ensemble import RandomForestClassifier
model2=RandomForestClassifier(class_weight='balanced')
model2.fit(x_train,y_train)
pred=model2.predict(x_test)
acc=accuracy_score(y_test,pred)
print(acc)
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))



In [ ]:
#GradientBoosting
from sklearn.ensemble import GradientBoostingClassifier
model3=GradientBoostingClassifier()
model3.fit(x_train,y_train)
pred=model3.predict(x_test)
acc=accuracy_score(y_test,pred)
print(acc)
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))



In [ ]:
#XGBoosting
from xgboost import XGBClassifier
model4=XGBClassifier(scale_pos_weight=60)
model4.fit(x_train,y_train)
pred=model4.predict(x_test)
acc=accuracy_score(y_test,pred)
print(acc)
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))



The model is performing reasonably well on Class 0 but has low recall for Class 1 (50%), meaning it misses half of the actual Class 1 observations. Since the classes are imbalanced (1041 vs 366), the model may be biased toward the majority class. I would examine the confusion matrix, ROC-AUC, Precision-Recall curve, and training performance before concluding whether the issue is underfitting or class imbalance. Potential remedies include SMOTE, class weighting, feature engineering, and threshold optimization.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_prob = model4.predict_proba(x_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

auc = roc_auc_score(y_test, y_prob)

print("AUC Score:", auc)

plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0,1], [0,1], '--')

plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve')

plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

models = {"Model":model,
    "Model1": model1,
    "Model2": model2,
    "Model3": model3,
    "Model4": model4
}

plt.figure(figsize=(8,6))

for name, model in models.items():

    y_prob = model.predict_proba(x_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    auc = roc_auc_score(y_test, y_prob)

    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1], [0,1], 'k--', label='Random')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')

plt.legend(loc='lower right')
plt.grid(True)

plt.show()

#some important hyperparameter of these models
*LogisticRegression
LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    class_weight=None
)
| Parameter      | Meaning                           |
| -------------- | --------------------------------- |
| `C`            | Regularization strength (inverse) |
| `penalty`      | l1, l2, elasticnet                |
| `solver`       | Optimization algorithm            |
| `max_iter`     | Number of iterations              |
| `class_weight` | Handle class imbalance            |


* Decision Tree
DecisionTreeClassifier(
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None
)
| Parameter           | Meaning                      |
| ------------------- | ---------------------------- |
| `criterion`         | gini / entropy               |
| `max_depth`         | Maximum tree depth           |
| `min_samples_split` | Minimum samples to split     |
| `min_samples_leaf`  | Minimum samples in leaf      |
| `max_features`      | Features considered at split |

* RandomForestClassifier
  
RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True
)
| Parameter           | Meaning                   |
| ------------------- | ------------------------- |
| `n_estimators`      | Number of trees           |
| `max_depth`         | Maximum depth             |
| `max_features`      | Features per split        |
| `min_samples_split` | Split threshold           |
| `min_samples_leaf`  | Leaf size                 |
| `bootstrap`         | Sampling with replacement |


* GradientBoostingClassifier
GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0)
| Parameter           | Meaning                   |
| ------------------- | ------------------------- |
| `n_estimators`      | Number of boosting rounds |
| `learning_rate`     | Shrinkage factor          |
| `max_depth`         | Tree depth                |
| `subsample`         | Row sampling              |
| `min_samples_split` | Split threshold           |

** XGBClassifier
XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1)
| Parameter          | Meaning                |
| ------------------ | ---------------------- |
| `n_estimators`     | Number of trees        |
| `learning_rate`    | Boosting step size     |
| `max_depth`        | Tree depth             |
| `subsample`        | Row sampling           |
| `colsample_bytree` | Feature sampling       |
| `gamma`            | Minimum loss reduction |
| `reg_alpha`        | L1 regularization      |
| `reg_lambda`       | L2 regularization      |
| `min_child_weight` | Minimum leaf weight    |


In [ ]:
df['Churn'].value_counts()

***Make Imblanced Data Blanced***
# Apply SMOTE only on the training dataset to balance churn and non-churn classes.
# This prevents the model from becoming biased toward the majority class.
# Note:SMOTE should never be applied before train-test split,as it may introduce data leakage.

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=99)

X_smote, y_smote = smote.fit_resample(x, y)

In [ ]:
import pandas as pd

y_smote['Churn'].value_counts()

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X_smote,y_smote)

In [ ]:
x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])

***Hyper Prameter TUning***
# Optimize model parameters using Grid Search
# or Randomized Search with Cross Validation
# to improve generalization performance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='roc_auc'
)

grid.fit(x_train, y_train)

print(grid.best_params_)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100,200,500],
    'max_depth': [5,10,None],
    'min_samples_split': [2,5,10],
    'max_features': ['sqrt','log2']
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(x_train, y_train)
print(grid.best_params_)

In [ ]:
print(grid.best_params_)

In [ ]:
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [100,200,500],
    'learning_rate': [0.01,0.05,0.1],
    'max_depth': [3,5,7],
    'subsample': [0.8,1.0],
    'colsample_bytree': [0.8,1.0]
}

grid = GridSearchCV(
    XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(x_train, y_train)
print(grid.best_params_)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

param_grid = {
    'n_estimators': [100,200],
    'learning_rate': [0.01,0.05,0.1],
    'max_depth': [3,5],
    'subsample': [0.8,1.0]
}

grid = GridSearchCV(
    GradientBoostingClassifier(),
    param_grid,
    cv=5,
    scoring='roc_auc'
)

grid.fit(x_train, y_train)

In [ ]:
print(grid.best_params_)

In [ ]:
lr={'C': 100, 'penalty': 'l2', 'solver': 'liblinear'}
rf={'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 500}
gb={'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
xgb={'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


# Identify the most influential features
# affecting customer churn.
# These insights can guide business teams
# in designing customer retention strategies.

In [ ]:
model1 = LogisticRegression(
    C=100,
    penalty='l2',
    solver='liblinear',
    max_iter=1000,
    random_state=42
)
model1.fit(x_train,y_train)
pred=model1.predict(x_test)
print(accuracy_score(pred,y_test))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

In [ ]:
model2 = RandomForestClassifier(
    max_depth=None,
    max_features='sqrt',
    min_samples_split=5,
    n_estimators=500,
    random_state=42
)
model2.fit(x_train,y_train)
pred=model2.predict(x_test)
print(accuracy_score(pred,y_test))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

In [ ]:
model3 = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=5,
    n_estimators=200,
    subsample=0.8,
    random_state=42
)
model3.fit(x_train,y_train)
pred=model3.predict(x_test)
print(accuracy_score(pred,y_test))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

In [ ]:
model4 = XGBClassifier(
    colsample_bytree=0.8,
    learning_rate=0.01,
    max_depth=3,
    n_estimators=100,
    subsample=0.8,
    eval_metric='logloss',
    random_state=42
)
model4.fit(x_train,y_train)
pred=model4.predict(x_test)
print(accuracy_score(pred,y_test))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

# Evaluate each model using multiple metrics:
# • Accuracy
# • Precision
# • Recall
# • F1 Score
# • ROC-AUC
# Since churn prediction is an imbalanced classification problem,
# Recall and F1-score are more informative than Accuracy alone.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

models = {"Logistic Regression":model1,
    "RF": model2,
    "GB": model3,
    "XGB": model4
}

plt.figure(figsize=(8,6))

for name, model in models.items():

    y_prob = model.predict_proba(x_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    auc = roc_auc_score(y_test, y_prob)

    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1], [0,1], 'k--', label='Random')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')

plt.legend(loc='lower right')
plt.grid(True)

plt.show()

In [ ]:
#GB is the best model

precesion recall
confusion matrix
bias variance 
underfitting overfitting
auc roc
loss function
hyperparameter tuning
etc

# Key Business Insights:
#
# • Customers with month-to-month contracts show the highest churn rate.
# • Customers with shorter tenure are significantly more likely to churn.
# • Higher monthly charges are associated with increased churn.
# • Electronic check payment customers churn more frequently.
# • Long-term contracts greatly improve customer retention.

# Conclusion:
#
# The analysis identified several important factors
# influencing customer churn.
#
# Machine learning models successfully learned customer behavior,
# allowing the business to predict potential churners.
#
# These predictions can support proactive retention strategies,
# reduce customer loss, and improve long-term revenue.